## Cell 1：参数

In [ ]:
from pathlib import Path

NUM_QUESTIONS = 200
RANDOM_SEED = 42
PROJECT_ROOT = Path('/content/hotpot_graphrag')
VENV_ROOT = Path('/content/graphrag_env')
VENV_PYTHON = VENV_ROOT / 'bin' / 'python'
VENV_GRAPHRAG = VENV_ROOT / 'bin' / 'graphrag'
GRAPHRAG_VERSION = '2.7.2'
CHAT_MODEL = 'gpt-4o-mini'
EMBEDDING_MODEL = 'text-embedding-3-small'
print(NUM_QUESTIONS, PROJECT_ROOT, VENV_ROOT)

200 /content/hotpot_graphrag /content/graphrag_env


## Cell 2：创建隔离环境并安装依赖

In [ ]:
import shutil, subprocess

if VENV_ROOT.exists():
    shutil.rmtree(VENV_ROOT)

subprocess.run(['python','-m','pip','install','-q','virtualenv'], check=True)
subprocess.run(['python','-m','virtualenv',str(VENV_ROOT)], check=True)

p = subprocess.run([
    str(VENV_PYTHON), '-m', 'pip', 'install',
    f'graphrag=={GRAPHRAG_VERSION}', 'datasets>=3,<5', 'openai>=1.40,<3'
], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(p.stdout)
if p.returncode != 0:
    raise RuntimeError('依赖安装失败')

INFO: pip is looking at multiple versions of scipy to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 9.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 14.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 11.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 16.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 15.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 18.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 19.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 21.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 21.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 917.8/917.8 kB 15.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Cell 3：检查隔离环境

In [ ]:
import subprocess
code = "from importlib.metadata import version; import graphrag,datasets,numpy,pandas,openai; print('GraphRAG',version('graphrag')); print('Datasets',datasets.__version__); print('NumPy',numpy.__version__); print('Pandas',pandas.__version__); print('OpenAI',openai.__version__)"
p = subprocess.run([str(VENV_PYTHON),'-P','-c',code], cwd='/tmp', text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(p.stdout)
if p.returncode != 0:
    raise RuntimeError('环境检查失败')

GraphRAG 2.7.2
Datasets 4.8.5
NumPy 1.26.4
Pandas 2.3.3
OpenAI 2.54.0



## Cell 4：输入 OpenAI API Key

In [ ]:
from getpass import getpass
OPENAI_API_KEY_VALUE = getpass('请输入 OpenAI API Key: ').strip()
if not OPENAI_API_KEY_VALUE:
    raise ValueError('API Key 不能为空')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

请输入 OpenAI API Key: ··········


## Cell 5：准备 HotpotQA hard + bridge 共享语料

In [ ]:
from pathlib import Path
prepare_script = r"""
import argparse,json,re
from pathlib import Path
from datasets import load_dataset
p=argparse.ArgumentParser(); p.add_argument('--root'); p.add_argument('--n',type=int); p.add_argument('--seed',type=int); a=p.parse_args()
root=Path(a.root); inp=root/'input'; data=root/'experiment_data'; inp.mkdir(parents=True,exist_ok=True); data.mkdir(parents=True,exist_ok=True)
for f in inp.glob('*.txt'): f.unlink()
ds=load_dataset('hotpotqa/hotpot_qa','distractor',split='validation')
ds=ds.filter(lambda x:x['level']=='hard' and x['type']=='bridge').shuffle(seed=a.seed).select(range(min(a.n,len(ds))))
docs={}; qs=[]
for r in ds:
    qs.append({'id':r['id'],'question':r['question'],'answer':r['answer'],'supporting_facts':r['supporting_facts']})
    for title,sents in zip(r['context']['title'],r['context']['sentences']):
        txt=' '.join(s.strip() for s in sents if s.strip()).strip()
        if len(txt)>len(docs.get(title,'')): docs[title]=txt
for i,(title,txt) in enumerate(sorted(docs.items())):
    name=re.sub(r'[^A-Za-z0-9_-]+','_',title).strip('_') or 'document'
    (inp/f'{i:06d}_{name[:100]}.txt').write_text(f'Title: {title}\n\n{txt}\n',encoding='utf-8')
(data/'questions.json').write_text(json.dumps(qs,ensure_ascii=False,indent=2),encoding='utf-8')
print('questions',len(qs)); print('documents',len(docs)); print('input',inp)
"""
Path('/content/prepare_hotpot.py').write_text(prepare_script,encoding='utf-8')
p = subprocess.run([str(VENV_PYTHON),'-P','/content/prepare_hotpot.py','--root',str(PROJECT_ROOT),'--n',str(NUM_QUESTIONS),'--seed',str(RANDOM_SEED)], cwd='/tmp', text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(p.stdout)
if p.returncode != 0: raise RuntimeError('数据准备失败')


Generating train split: 100%|██████████| 90447/90447 [00:05<00:00, 16515.23 examples/s]

Generating validation split: 100%|██████████| 7405/7405 [00:00<00:00, 34234.48 examples/s]

Filter: 100%|██████████| 7405/7405 [00:00<00:00, 46298.96 examples/s]
questions 200
documents 1986
input /content/hotpot_graphrag/input



## Cell 6：初始化 GraphRAG

In [ ]:
import os, subprocess
from pathlib import Path
SAFE_CWD=Path('/tmp/graphrag_runner'); SAFE_CWD.mkdir(parents=True,exist_ok=True)
GRAPHRAG_ENV=os.environ.copy(); GRAPHRAG_ENV['PYTHONSAFEPATH']='1'; GRAPHRAG_ENV['MPLBACKEND']='Agg'
cmd=[str(VENV_PYTHON),'-P',str(VENV_GRAPHRAG),'init','--root',str(PROJECT_ROOT),'--force']
p=subprocess.run(cmd,env=GRAPHRAG_ENV,cwd=str(SAFE_CWD),text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(p.stdout)
if p.returncode != 0: raise RuntimeError('GraphRAG 初始化失败')

## Cell 7：写入 Key，并修正模型配置

In [ ]:
(PROJECT_ROOT/'.env').write_text(f'GRAPHRAG_API_KEY={OPENAI_API_KEY_VALUE}\n',encoding='utf-8')
settings=PROJECT_ROOT/'settings.yaml'
text=settings.read_text(encoding='utf-8')
text=text.replace('gpt-4-turbo-preview',CHAT_MODEL).replace('text-embedding-ada-002',EMBEDDING_MODEL)
settings.write_text(text,encoding='utf-8')
for i,line in enumerate(text.splitlines(),1):
    if 'model:' in line.lower() or 'model_id:' in line.lower(): print(i,line)
assert 'gpt-4-turbo-preview' not in text

8   default_chat_model:
13     model: gpt-4o-mini
23   default_embedding_model:
28     model: text-embedding-3-small
76   model_id: default_embedding_model
80   model_id: default_chat_model
86   model_id: default_chat_model
100   model_id: default_chat_model
106   model_id: default_chat_model
127   chat_model_id: default_chat_model
128   embedding_model_id: default_embedding_model
132   chat_model_id: default_chat_model
138   chat_model_id: default_chat_model
139   embedding_model_id: default_embedding_model
144   chat_model_id: default_chat_model
145   embedding_model_id: default_embedding_model


## Cell 8：模型预检（必须成功后再索引）

In [ ]:
code=f"""
import os
from openai import OpenAI
c=OpenAI(api_key=os.environ['GRAPHRAG_API_KEY'])
r=c.chat.completions.create(model={CHAT_MODEL!r},messages=[{{'role':'user','content':'Reply only with OK'}}],temperature=0,max_tokens=5)
e=c.embeddings.create(model={EMBEDDING_MODEL!r},input='test')
print('chat',r.choices[0].message.content)
print('embedding_dimensions',len(e.data[0].embedding))
"""
env=GRAPHRAG_ENV.copy(); env['GRAPHRAG_API_KEY']=OPENAI_API_KEY_VALUE
p=subprocess.run([str(VENV_PYTHON),'-P','-c',code],env=env,cwd=str(SAFE_CWD),text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(p.stdout)
if p.returncode != 0: raise RuntimeError('模型预检失败，不要运行下一格')

chat OK
embedding_dimensions 1536



## Cell 9：建立 GraphRAG 标准索引（实时日志）

In [ ]:
import time
cmd=[str(VENV_PYTHON),'-P',str(VENV_GRAPHRAG),'index','--root',str(PROJECT_ROOT),'--method','standard','--verbose']
print(' '.join(cmd)); print('='*100)
start=time.time()
p=subprocess.Popen(cmd,env=GRAPHRAG_ENV,cwd=str(SAFE_CWD),text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
assert p.stdout is not None
for line in p.stdout: print(line,end='')
rc=p.wait(); print('='*100); print('returncode',rc); print('minutes',(time.time()-start)/60)
if rc != 0: raise RuntimeError('GraphRAG 索引失败')

流式输出内容被截断，只能显示最后 5000 行内容。
  23896 / 25850 ............................................................................
  23897 / 25850 ............................................................................
  23898 / 25850 ............................................................................
  23899 / 25850 ............................................................................
  23900 / 25850 ............................................................................
  23901 / 25850 ............................................................................
  23902 / 25850 ............................................................................
  23903 / 25850 ............................................................................
  23904 / 25850 ............................................................................
  23905 / 25850 ............................................................................
  23906 / 25850 ...........................

## Cell 10：检查索引文件

In [ ]:
out=PROJECT_ROOT/'output'
if not out.exists(): raise FileNotFoundError(out)
files=[p.relative_to(PROJECT_ROOT) for p in out.rglob('*') if p.is_file()]
print('files',len(files))
for f in sorted(files)[:50]: print(f)

files 122
output/communities.parquet
output/community_reports.parquet
output/context.json
output/documents.parquet
output/entities.parquet
output/lancedb/default-community-full_content.lance/_indices/d94e9851-bc3a-4910-99e7-d845546724ff/auxiliary.idx
output/lancedb/default-community-full_content.lance/_indices/d94e9851-bc3a-4910-99e7-d845546724ff/index.idx
output/lancedb/default-community-full_content.lance/_transactions/0-7c5813e9-8941-4d11-9cf4-322512fcc950.txn
output/lancedb/default-community-full_content.lance/_transactions/1-634d0653-3790-4951-80ab-563bd75f944f.txn
output/lancedb/default-community-full_content.lance/_transactions/2-6a4202ab-7968-42e5-8240-9af288b8610e.txn
output/lancedb/default-community-full_content.lance/_transactions/3-5c87ffb1-b7c3-4e2d-a203-a71669c8a550.txn
output/lancedb/default-community-full_content.lance/_transactions/4-348585a3-5b52-4e1f-bd55-cae3d788f940.txn
output/lancedb/default-community-full_content.lance/_versions/18446744073709551610.manifest
outp

## Cell 11：测试一道 Local Search

In [ ]:
import json
import subprocess

questions = json.loads(
    (PROJECT_ROOT / "experiment_data" / "questions.json")
    .read_text(encoding="utf-8")
)

q = questions[0]

cmd = [
    str(VENV_PYTHON),
    "-P",
    str(VENV_GRAPHRAG),
    "query",
    "--root",
    str(PROJECT_ROOT),
    "--method",
    "local",
    "--response-type",
    "Single short answer without explanation",
    "--query",
    q["question"],
]

p = subprocess.run(
    cmd,
    env=GRAPHRAG_ENV,
    cwd=str(SAFE_CWD),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print("Question:", q["question"])
print("Gold:", q["answer"])
print(p.stdout)

if p.returncode != 0:
    raise RuntimeError("Local Search 失败")

Question: What Dutch painter specialized in domestic interior scenes of middle class life and has a painting of a young woman dressed in a yellow shawl in the Louvre, Paris?
Gold: Johannes Vermeer
## Dutch Painter Specializing in Domestic Scenes

The Dutch painter you are referring to is **Johannes Vermeer**. He is renowned for his exceptional contributions to art, particularly in depicting domestic interior scenes of middle-class life. One of his notable paintings, **The Lacemaker**, features a young woman dressed in a yellow shawl and is housed in the Louvre, Paris [Data: Reports (682); Entities (4399, 5538); Relationships (5302, 5303)].



In [ ]:
pred_path = (
    PROJECT_ROOT
    / "experiment_data"
    / "graphrag_local_predictions.jsonl"
)

print("文件是否存在:", pred_path.exists())

if pred_path.exists():
    print(
        "旧记录数:",
        len(
            pred_path.read_text(
                encoding="utf-8"
            ).splitlines()
        ),
    )
    pred_path.unlink()
    print("旧预测文件已删除")

文件是否存在: False


In [ ]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY_VALUE)

def shorten_answer(question, long_answer):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": f"""
Question:
{question}

Long answer:
{long_answer}

Return only the shortest final answer.
Do not explain.
Do not include citations.
""",
            }
        ],
        temperature=0,
        max_tokens=20,
    )

    return response.choices[0].message.content.strip()

## Cell 12：批量预测（断点续跑）

In [ ]:
import json,time
pred_path=PROJECT_ROOT/'experiment_data'/'graphrag_local_predictions.jsonl'
completed=set()
if pred_path.exists():
    for line in pred_path.read_text(encoding='utf-8').splitlines():
        try: completed.add(json.loads(line)['id'])
        except: pass
for i,item in enumerate(questions,1):
    if item['id'] in completed: continue
    cmd = [
    str(VENV_PYTHON),
    "-P",
    str(VENV_GRAPHRAG),
    "query",
    "--root",
    str(PROJECT_ROOT),
    "--method",
    "local",
    "--response-type",
    "Single short answer without explanation",
    "--query",
    item["question"],
]
    t=time.time(); p=subprocess.run(cmd,env=GRAPHRAG_ENV,cwd=str(SAFE_CWD),text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    lines=[x.strip() for x in p.stdout.splitlines() if x.strip()]
    #prediction=lines[-1] if lines else ''
    long_answer = lines[-1] if lines else ""

    prediction = shorten_answer(
    item["question"],
    long_answer,
)
    rec={'id':item['id'],'question':item['question'],'gold_answer':item['answer'],'prediction':prediction,'returncode':p.returncode,'seconds':time.time()-t,'raw_output':p.stdout}
    with pred_path.open('a',encoding='utf-8') as f: f.write(json.dumps(rec,ensure_ascii=False)+'\n')
    print(f'[{i}/{len(questions)}] rc={p.returncode} pred={prediction!r}')
print(pred_path)

[1/200] rc=0 pred='Johannes Vermeer'
[2/200] rc=0 pred='11½ miles (19 km)'
[3/200] rc=0 pred='Benjamin Cheever'
[4/200] rc=0 pred='Woody Woodpecker'
[5/200] rc=0 pred="I don't have that information."
[6/200] rc=0 pred='The 36th Chamber of Shaolin'
[7/200] rc=0 pred='Wichita, Kansas'
[8/200] rc=0 pred='Anderson Silva'
[9/200] rc=0 pred='Stephen King'
[10/200] rc=0 pred='Chloë Agnew'
[11/200] rc=0 pred='2004'
[12/200] rc=0 pred='1,472 kilometers'
[13/200] rc=0 pred='Ireland, Scotland, Wales, Cornwall, Brittany, Isle of Man.'
[14/200] rc=0 pred='Olmsted Falls, Ohio'
[15/200] rc=0 pred='To SquarePants or Not to SquarePants'
[16/200] rc=0 pred='Captain Hans Geering'
[17/200] rc=0 pred='June 6, 1959'
[18/200] rc=0 pred='Canterbury'
[19/200] rc=0 pred='Caesars Entertainment Corporation'
[20/200] rc=0 pred='Urdu'
[21/200] rc=0 pred='Pope John X'
[22/200] rc=0 pred="Both songs are from Coldplay's album *Mylo Xyloto*."
[23/200] rc=0 pred='Crimean War'
[24/200] rc=0 pred='Birmingham City F.C.'
[2

## Cell 13：计算 Answer EM / F1

In [ ]:
import re,string,json
from collections import Counter

def norm(s):
    s=s.lower(); s=''.join(ch for ch in s if ch not in string.punctuation); s=re.sub(r'\b(a|an|the)\b',' ',s); return ' '.join(s.split())
def em(p,g): return float(norm(p)==norm(g))
def f1(p,g):
    pt,gt=norm(p).split(),norm(g).split()
    if not pt or not gt: return float(pt==gt)
    ov=sum((Counter(pt)&Counter(gt)).values())
    if ov==0:return 0.0
    pr=ov/len(pt); rc=ov/len(gt); return 2*pr*rc/(pr+rc)
rows=[json.loads(x) for x in pred_path.read_text(encoding='utf-8').splitlines() if x.strip()]
ok=[r for r in rows if r['returncode']==0]
for r in ok: r['em']=em(r['prediction'],r['gold_answer']); r['f1']=f1(r['prediction'],r['gold_answer'])
summary={'total':len(rows),'successful':len(ok),'failed':len(rows)-len(ok),'answer_em':sum(r['em'] for r in ok)/len(ok) if ok else 0,'answer_f1':sum(r['f1'] for r in ok)/len(ok) if ok else 0}
print(json.dumps(summary,ensure_ascii=False,indent=2))
(PROJECT_ROOT/'experiment_data'/'evaluation.json').write_text(json.dumps({'summary':summary,'records':rows},ensure_ascii=False,indent=2),encoding='utf-8')

{
  "total": 200,
  "successful": 200,
  "failed": 0,
  "answer_em": 0.48,
  "answer_f1": 0.6594274246994836
}


116155

## Cell 14：查看错误案例

In [ ]:
wrong=[r for r in ok if r.get('em',0)==0]
for r in wrong[:10]:
    print('='*100); print('Q:',r['question']); print('Gold:',r['gold_answer']); print('Pred:',r['prediction']); print('F1:',r['f1'])

Q: Aughton Park railway station is on a branch of the Northern Line how far from Liverpool?
Gold: 12+3/4 mi northeast of Liverpool
Pred: 11½ miles (19 km)
F1: 0.0
Q: Who is the son of the author of the short story "The Enormous Radio."
Gold: Benjamin Hale Cheever
Pred: Benjamin Cheever
F1: 0.8
Q: How many records did the singer who sang "Four Seasons of Love" sell worldwide?
Gold: 140 million
Pred: I don't have that information.
F1: 0.0
Q: Both Bishop Carroll Catholic High School and Kapaun Mt. Carmel Catholic High School are located in which city in Kansas?
Gold: Wichita
Pred: Wichita, Kansas
F1: 0.6666666666666666
Q: What is the name of the daughter of the Irish entertainer in the girl band Maxi, Dick and Twink?
Gold: Chloë Alexandra Adele Emily Agnew
Pred: Chloë Agnew
F1: 0.5714285714285715
Q: How long is the river for which Frenchmans Creek is a tributary?
Gold: 2844 km
Pred: 1,472 kilometers
F1: 0.0
Q: In which six Western European territories have Celtic languages or cultural tra